# Feedforward Neural Network

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

## Load the data

In [2]:
fp = "SPY_resistance.jsonl"

with open(fp, "r") as f:
    data = [json.loads(line) for line in f]

DATA = pd.DataFrame(data, columns=['hX', 'lX', 'hy', 'ly'])

In [3]:
# Use the last 20 rows for manual testing
df = DATA.copy().iloc[:-20]

X = df['hX']
y = df['hy']

In [4]:
# Extract features and targets
X = np.array(DATA['hX'].tolist()).astype(np.float32)
y = np.array(DATA['hy'].tolist()).astype(np.float32)

# Split the data into train and test sets
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)

## Train the model

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [6]:
class TimeSeriesDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


class TimeSeriesModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(TimeSeriesModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [8]:
# Create DataLoader instances
batch_size = 32

train_dataset = TimeSeriesDataset(train_X, train_y)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TimeSeriesDataset(test_X, test_y)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Instantiate the model, loss function, and optimizer
input_size = X.shape[1]
hidden_size = 64
output_size = 1

model = TimeSeriesModel(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    for batch_features, batch_targets in train_loader:
        # Forward pass
        outputs = model(batch_features)
        loss = criterion(outputs.squeeze(), batch_targets)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluate the model on the test set
model.eval()
total_loss = 0
with torch.no_grad():
    for batch_features, batch_targets in test_loader:
        outputs = model(batch_features)
        loss = criterion(outputs.squeeze(), batch_targets)
        total_loss += loss.item()

average_loss = total_loss / len(test_loader)
print(f'Average Test Loss: {average_loss:.4f}')

Epoch [1/20], Loss: 1154.0588
Epoch [2/20], Loss: 574.2137
Epoch [3/20], Loss: 158.3050
Epoch [4/20], Loss: 20.2594
Epoch [5/20], Loss: 35.0019
Epoch [6/20], Loss: 11.7612
Epoch [7/20], Loss: 9.5721
Epoch [8/20], Loss: 12.3001
Epoch [9/20], Loss: 15.3284
Epoch [10/20], Loss: 187.9253
Epoch [11/20], Loss: 31.6747
Epoch [12/20], Loss: 9.7144
Epoch [13/20], Loss: 52.0184
Epoch [14/20], Loss: 17.4951
Epoch [15/20], Loss: 63.1047
Epoch [16/20], Loss: 2.6694
Epoch [17/20], Loss: 19.5013
Epoch [18/20], Loss: 71.4856
Epoch [19/20], Loss: 5.0951
Epoch [20/20], Loss: 3.3123
Average Test Loss: 22.6426


## Predict Weekly Resistance

In [9]:
from axiom.config import DATA_DIR

DAILY_DATA = pd.read_csv(DATA_DIR.joinpath("daily", "SPY_2024-07-20.csv"))

In [10]:
df = DAILY_DATA.copy()

# Filter columns
columns = ["datetime", "high", "low"]
df = df[columns]

# Convert int to datetime
df["datetime"] = pd.to_datetime(df["datetime"], unit="ms")

# Add a new column for the day of the week, 0=Monday, 6=Sunday
df["day"] = df["datetime"].dt.dayofweek
df.head()

,datetime,high,low,day
0,2004-07-19 05:00:00,110.96,109.99,0
1,2004-07-20 05:00:00,111.90,110.25,1
2,2004-07-21 05:00:00,112.06,109.45,2
3,2004-07-22 05:00:00,110.39,108.77,3
4,2004-07-23 05:00:00,109.71,108.69,4


In [11]:
# In the last 10 rows, get the index of the last "4" day
last_friday_index = df[df["day"] == 4].index[-1]

# Get the highs and lows of the last 5 days ending on the last "4" day
last_days = df.loc[last_friday_index - 9: last_friday_index]

last_days_high = last_days["high"].values
last_days_low = last_days["low"].values
last_days_high, last_days_low

(array([556.2501, 557.18  , 561.67  , 562.33  , 563.67  , 564.8371,
        565.16  , 560.51  , 559.52  , 554.08  ]),
 array([554.19, 555.52, 556.77, 555.83, 557.15, 559.63, 562.1 , 556.61,
        550.43, 547.91]))

In [12]:
input_array = np.array(last_days_high, dtype=np.float32)
input_tensor = torch.tensor(input_array).unsqueeze(0)

# Set the model to evaluation mode
model.eval()

# Run inference
with torch.no_grad():
    prediction = model(input_tensor)
    predicted_value = prediction.item()

print(f'{df.iloc[last_friday_index]["datetime"]} Predicted: {predicted_value:.2f}')

2024-07-19 05:00:00 Predicted: 566.35
